# ETL scratch — stage-2 staging

Scratch-only. Production orchestration lives in `tools.etl.pipeline`,
invoked by the ARQ worker in `backend/app/etl/tasks.py`. This cell is
a debug entry point for running the obrigação pipeline against a narrow
filter and inspecting the resulting staging rows.

In [ ]:
import os
from datetime import date
from pathlib import Path

# Ensure CWD is repo root so relative paths and `tools.*` imports resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")

from langchain_openai import AzureChatOpenAI
from sqlalchemy.orm import sessionmaker

from tools.etl.pipeline import (
    ExtractionFilters,
    enqueue_obrigacao_extraction,
)
from tools.schema import Obrigacao, ResponsibleChoice
from tools.utils import DB_DECISOES, get_connection

gpt4 = AzureChatOpenAI(deployment_name="gpt-4-turbo", model_name="gpt-4")
extractor_obrigacao = gpt4.with_structured_output(Obrigacao, include_raw=False)
extractor_responsible = gpt4.with_structured_output(ResponsibleChoice, include_raw=False)

SessionLocal = sessionmaker(bind=get_connection(DB_DECISOES))
session = SessionLocal()
try:
    report = enqueue_obrigacao_extraction(
        ExtractionFilters(
            start_date=date(2026, 2, 1),
            end_date=date(2026, 2, 28),
            process_numbers=["011943/2012"],
        ),
        extractor=extractor_obrigacao,
        responsible_extractor=extractor_responsible,
        session=session,
    )
    print(report)
finally:
    session.close()